In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!kaggle competitions download -c store-sales-time-series-forecasting

  0% 0.00/21.4M [00:00<?, ?B/s]
100% 21.4M/21.4M [00:00<00:00, 1.97GB/s]


In [4]:
!unzip /content/store-sales-time-series-forecasting.zip

Archive:  /content/store-sales-time-series-forecasting.zip
  inflating: holidays_events.csv     
  inflating: oil.csv                 
  inflating: sample_submission.csv   
  inflating: stores.csv              
  inflating: test.csv                
  inflating: train.csv               
  inflating: transactions.csv        


In [5]:
# Load data
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [6]:
train_data.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [7]:
train_data.shape,test_data.shape

((3000888, 6), (28512, 5))

In [8]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


In [9]:
train_data.isna().sum()

,0
id,0
date,0
store_nbr,0
family,0
sales,0
onpromotion,0


In [10]:
# Convert date column ro datetime as it is object
train_data['date'] = pd.to_datetime(train_data['date'])
test_data['date'] = pd.to_datetime(test_data['date'])

In [11]:
print(f"Train Date Range: {train_data['date'].min()} to {train_data['date'].max()}")
print(f"Test Date Range: {test_data['date'].min()} to {test_data['date'].max()}")

Train Date Range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Test Date Range: 2017-08-16 00:00:00 to 2017-08-31 00:00:00


In [12]:
# Sort by date
train_data = train_data.sort_values(by=['date', 'store_nbr', 'family']).reset_index(drop=True)

## Time-Based Validation Split
We will use a time-based train/validation split rather than a random split.


In [13]:
# last 15 days data as val data as test data is also 15 days
val_start_date = train_data['date'].max() - pd.Timedelta(days=15)
val_start_date

Timestamp('2017-07-31 00:00:00')

In [14]:
train = train_data[train_data['date'] <= val_start_date]
val = train_data[train_data['date'] > val_start_date]

print(f"Train set: {train.shape[0]} rows")
print(f"Val set: {val.shape[0]} rows")

Train set: 2974158 rows
Val set: 26730 rows


## Baseline (Naïve Forecast)
Prediction = previous day's sales for the same store and family


In [15]:
# Naive Baseline: Use the previous day's value
# Merge validation with the latest available day from training set for each store & family
latest_train_sales = train[train['date'] == train['date'].max()][['store_nbr', 'family', 'sales']]
latest_train_sales.head()

,store_nbr,family,sales
2972376,1,AUTOMOTIVE,8.0
2972377,1,BABY CARE,0.0
2972378,1,BEAUTY,3.0
2972379,1,BEVERAGES,2414.0
2972380,1,BOOKS,1.0


In [16]:
# rename sales ===> naive_pred
latest_train_sales.rename(columns={'sales': 'naive_pred'}, inplace=True)

In [17]:
naive_val = val.merge(latest_train_sales, on=['store_nbr', 'family'], how='left')
naive_val['naive_pred'].fillna(0, inplace=True)

naive_rmse = np.sqrt(mean_squared_error(naive_val['sales'], naive_val['naive_pred']))
print(f"Naïve Forecast Validation RMSE: {naive_rmse:.2f}")

Naïve Forecast Validation RMSE: 363.87


## ARIMA / SARIMA Model


Each validation day uses the previous day's actual sales.

``y^​t​=yt−1​``

In [25]:
# Combine train + val temporarily
full_data = pd.concat([train, val]).sort_values(
    ['store_nbr', 'family', 'date']
)

# Create lag-1 feature
full_data['lag_1'] = full_data.groupby(
    ['store_nbr', 'family']
)['sales'].shift(1)

# Extract validation rows
naive_val = full_data[full_data['date'] > val_start_date]

# Compute RMSE
naive_rmse = np.sqrt(mean_squared_error(
    naive_val['sales'],
    naive_val['lag_1']
))

print(f"Proper Rolling Naïve RMSE: {naive_rmse:.2f}")

Proper Rolling Naïve RMSE: 368.80


## Prophet Model


In [27]:
from prophet import Prophet

daily_train = train.groupby('date')['sales'].sum().reset_index()
daily_val = val.groupby('date')['sales'].sum().reset_index()


# Prepare data
prophet_train = daily_train.rename(columns={'date': 'ds', 'sales': 'y'})
prophet_val = daily_val.rename(columns={'date': 'ds', 'sales': 'y'})

# Fit model
model_prophet = Prophet()
model_prophet.fit(prophet_train)

# Predict ONLY on validation dates
forecast = model_prophet.predict(prophet_val[['ds']])

# Extract predictions
prophet_preds = forecast['yhat'].values

# Compute RMSE
prophet_rmse = np.sqrt(mean_squared_error(
    prophet_val['y'],
    prophet_preds
))

print(f"Correct Prophet RMSE: {prophet_rmse:.2f}")

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Correct Prophet RMSE: 94867.81


## 6. Machine Learning Models (RF, XGBoost, LightGBM)


In [20]:
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# Prepare features
features = ['store_nbr', 'onpromotion']

# Encode Family
le = LabelEncoder()
train['family_encoded'] = le.fit_transform(train['family'])
val['family_encoded'] = le.transform(val['family'])
features.append('family_encoded')

# Time-based features
for df in [train, val]:
    df['dayofweek'] = df['date'].dt.dayofweek
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month

features.extend(['dayofweek', 'day', 'month'])

X_train = train[features].fillna(0)
y_train = train['sales']
X_val = val[features].fillna(0)
y_val = val['sales']


In [21]:
# Random Forest (Subsetting data to train faster)
sample_idx = np.random.choice(len(X_train), size=100000, replace=False)
rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
rf.fit(X_train.iloc[sample_idx], y_train.iloc[sample_idx])
rf_preds = rf.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_preds))
print(f"Random Forest RMSE: {rf_rmse:.2f}")

Random Forest RMSE: 397.72


In [22]:
# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_val)
xgb_preds = np.maximum(xgb_preds, 0)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_preds))
print(f"XGBoost RMSE: {xgb_rmse:.2f}")

XGBoost RMSE: 377.53


In [23]:
# LightGBM
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
lgb_model.fit(X_train, y_train)
lgb_preds = lgb_model.predict(X_val)
lgb_preds = np.maximum(lgb_preds, 0)
lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_preds))
print(f"LightGBM RMSE: {lgb_rmse:.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.212631 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 346
[LightGBM] [Info] Number of data points in the train set: 2974158, number of used features: 6
[LightGBM] [Info] Start training from score 356.810778
LightGBM RMSE: 404.65


## Results Comparison


In [24]:
results = pd.DataFrame({
    'Model': ['Naive', 'Random Forest', 'XGBoost', 'LightGBM'],
    'Validation RMSE': [naive_rmse, rf_rmse, xgb_rmse, lgb_rmse]
})
print("Detailed Model vs RMSE (Granular, store-item level):")
print(results.sort_values(by='Validation RMSE'))

print("\nProphet and ARIMA were run structurally on aggregate level to avoid 1700+ individual models here.")



Detailed Model vs RMSE (Granular, store-item level):
           Model  Validation RMSE
0          Naive       363.871527
2        XGBoost       377.527316
1  Random Forest       397.718958
3       LightGBM       404.647317

Prophet and ARIMA were run structurally on aggregate level to avoid 1700+ individual models here.
